# EDA — WLASL Dataset (Top-100 Words)
**SignBridge | Day 1**

Explores the WLASL processed dataset — word-level ASL video clips.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from google.cloud import storage
import io

BUCKET      = 'signbridge-data'
WLASL_PREFIX = 'raw/wlasl/wlasl_data/'
FIGURES_DIR  = '../docs/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

client = storage.Client()
bucket = client.bucket(BUCKET)
print('GCS connected:', BUCKET)

## 1. Load nslt_100.json — Top 100 Words Subset

In [ ]:
# Load the top-100 split file from GCS
blob = bucket.blob(WLASL_PREFIX + 'nslt_100.json')
nslt_100 = json.loads(blob.download_as_text())

print(f'Type: {type(nslt_100)}')
if isinstance(nslt_100, dict):
    print(f'Keys: {list(nslt_100.keys())[:5]}')
elif isinstance(nslt_100, list):
    print(f'Length: {len(nslt_100)}')
    print(f'First entry: {nslt_100[0]}')

In [ ]:
# Load full WLASL annotation file
blob_full = bucket.blob(WLASL_PREFIX + 'WLASL_v0.3.json')
wlasl_data = json.loads(blob_full.download_as_text())

print(f'Total glosses in full dataset: {len(wlasl_data)}')
print(f'Sample entry keys: {list(wlasl_data[0].keys())}')
print(f'Sample gloss: {wlasl_data[0]["gloss"]}')
print(f'Sample instance keys: {list(wlasl_data[0]["instances"][0].keys())}')

## 2. Build Top-100 Gloss List & Clip Counts

In [ ]:
# Get the set of video IDs in nslt_100
if isinstance(nslt_100, dict):
    # nslt_100 maps video_id -> {action, start, end, split}
    top100_ids = set(nslt_100.keys())
else:
    top100_ids = set(str(x) for x in nslt_100)

print(f'Video IDs in top-100 split: {len(top100_ids)}')

# Map each gloss to its clip count within top-100
gloss_clips = defaultdict(int)
gloss_signers = defaultdict(set)

for entry in wlasl_data:
    gloss = entry['gloss']
    for inst in entry['instances']:
        vid_id = str(inst['video_id'])
        if vid_id in top100_ids:
            gloss_clips[gloss] += 1
            gloss_signers[gloss].add(inst.get('signer_id', 'unknown'))

print(f'Unique glosses in top-100: {len(gloss_clips)}')
print(f'Total clips: {sum(gloss_clips.values())}')

In [ ]:
# Sort by clip count
sorted_glosses = sorted(gloss_clips.items(), key=lambda x: x[1], reverse=True)
glosses  = [g for g, _ in sorted_glosses]
counts   = [c for _, c in sorted_glosses]

print('Top 10 glosses by clip count:')
for g, c in sorted_glosses[:10]:
    print(f'  {g}: {c} clips, {len(gloss_signers[g])} signers')

In [ ]:
# Plot clip distribution
fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(range(len(glosses)), counts, color='coral', edgecolor='white')
ax.set_xticks(range(len(glosses)))
ax.set_xticklabels(glosses, rotation=90, fontsize=7)
ax.set_xlabel('ASL Gloss (Word)')
ax.set_ylabel('Clip Count')
ax.set_title('WLASL Top-100 — Clip Count Per Gloss')
ax.axhline(np.mean(counts), color='blue', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'wlasl_clip_dist.png', dpi=150)
plt.show()
print('Saved: docs/figures/wlasl_clip_dist.png')

## 3. Frame Count Distribution (Sample Videos)

In [ ]:
# Check how many video files actually landed in GCS
video_blobs = list(client.list_blobs(BUCKET, prefix=WLASL_PREFIX + 'videos/'))
video_files = [b.name.split('/')[-1] for b in video_blobs if b.name.endswith('.mp4')]
print(f'Video files in GCS: {len(video_files)}')
print(f'Sample files: {video_files[:5]}')

In [ ]:
# Use cv2 to check frame counts on a sample of 50 videos
import cv2
import tempfile

sample_videos = video_files[:50]
frame_counts = []

for vid_file in sample_videos:
    blob_path = WLASL_PREFIX + 'videos/' + vid_file
    blob = bucket.blob(blob_path)
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        blob.download_to_filename(tmp.name)
        cap = cv2.VideoCapture(tmp.name)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_counts.append(frames)
        cap.release()
        os.unlink(tmp.name)

print(f'Frame count stats (sample of {len(frame_counts)} videos):')
print(f'  Min:    {min(frame_counts)}')
print(f'  Max:    {max(frame_counts)}')
print(f'  Mean:   {np.mean(frame_counts):.1f}')
print(f'  Median: {np.median(frame_counts):.1f}')

In [ ]:
# Plot frame count distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(frame_counts, bins=20, color='steelblue', edgecolor='white')
ax.axvline(30, color='red', linestyle='--', label='Target: 30 frames')
ax.set_xlabel('Frame Count')
ax.set_ylabel('Number of Videos')
ax.set_title('WLASL — Frame Count Distribution (Sample 50 Videos)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'wlasl_frame_dist.png', dpi=150)
plt.show()
print('Saved: docs/figures/wlasl_frame_dist.png')

## 4. Signer Diversity Check

In [ ]:
signer_counts = [len(v) for v in gloss_signers.values()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(signer_counts, bins=15, color='coral', edgecolor='white')
ax.set_xlabel('Number of Unique Signers')
ax.set_ylabel('Number of Glosses')
ax.set_title('WLASL Top-100 — Signer Diversity Per Gloss')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'wlasl_signer_diversity.png', dpi=150)
plt.show()

print(f'Avg signers per gloss: {np.mean(signer_counts):.1f}')
print(f'Min signers: {min(signer_counts)}, Max signers: {max(signer_counts)}')

## 5. MediaPipe Coverage Test on 1 Sample Video

In [ ]:
import mediapipe as mp
import tempfile

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils

# Download 1 sample video
sample_vid = video_files[0]
blob = bucket.blob(WLASL_PREFIX + 'videos/' + sample_vid)

with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
    blob.download_to_filename(tmp.name)
    tmp_path = tmp.name

cap = cv2.VideoCapture(tmp_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
detected_frames = 0
landmark_shapes = []

with mp_hands.Hands(static_image_mode=False, max_num_hands=2,
                    min_detection_confidence=0.5) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        if result.multi_hand_landmarks:
            detected_frames += 1
            lm = result.multi_hand_landmarks[0].landmark
            coords = np.array([[l.x, l.y, l.z] for l in lm])
            landmark_shapes.append(coords.shape)

cap.release()
os.unlink(tmp_path)

coverage = detected_frames / total_frames * 100
print(f'Video: {sample_vid}')
print(f'Total frames:    {total_frames}')
print(f'Detected frames: {detected_frames} ({coverage:.1f}%)')
print(f'Landmark shape per frame: {landmark_shapes[0] if landmark_shapes else "N/A"}')
assert landmark_shapes[0] == (21, 3), 'Expected (21, 3) landmarks per frame'
print('Landmark shape assertion passed: (21, 3) ✓')

## 6. EDA Findings Summary

In [ ]:
print('=== WLASL EDA FINDINGS ===')
print(f'Unique glosses (top-100):  {len(gloss_clips)}')
print(f'Total clips:               {sum(gloss_clips.values())}')
print(f'Min clips per gloss:       {min(counts)}')
print(f'Max clips per gloss:       {max(counts)}')
print(f'Mean clips per gloss:      {np.mean(counts):.1f}')
print(f'Avg signers per gloss:     {np.mean(signer_counts):.1f}')
print(f'Video files in GCS:        {len(video_files)}')
print(f'Median frame count:        {np.median(frame_counts):.0f}')
print(f'MediaPipe coverage:        {coverage:.1f}% (sample video)')
print('')
print('Notes for report_log.md:')
print('  - Class imbalance present: apply class weights during training')
print('  - Median frames != 30: pad/truncate to 30 during preprocessing')
print('  - MediaPipe misses some frames: zero-fill missing detections')